# Confidence-Aware Grape Leaf Disease Recognition — Baseline + Calibration

**COMP 2035 · Bryce Daniliuc**

---

### What this notebook produces

| Step | Output |
|---|---|
| 1. Dataset audit | Image counts, source-leaf counts, augmentation breakdown, leakage figures |
| 2. Partitioning | A split in which no source leaf spans two partitions |
| 3. Baseline | ResNet-50 trained with the protocol of Canoski et al. (2026) |
| 4. Calibration | Expected Calibration Error, reliability diagram, confidence histogram |
| 5. Temperature scaling | The single-parameter post-hoc correction of Guo et al. (2017) |

**Runtime** ≈ 25–40 min on a Colab T4. Set **Runtime → Change runtime type → T4 GPU** first.

---

### Method note — read before running

The supplied dataset contains **4,062 original photographs and 4,965 augmented copies**
(`_flipLR`, `_90deg`, `_180deg`, `_270deg`, `_new30degFlipLR`). Augmentation was applied
*before* the train/test folders were created, so copies of the same leaf appear on both
sides of the supplied split.

This notebook therefore does two things that a naive pipeline does not:

1. **Partitions at the source-leaf level** — every copy of a leaf lands in one partition.
2. **Evaluates on original photographs only** — augmented copies are used for training and
   are never scored. Augmented copies belonging to a validation or test leaf are discarded
   rather than moved to training, which would reintroduce the leak.

`SPLIT_MODE` lets you reproduce the alternatives for comparison. See Section 6.

### Deviations from the published protocol — be ready to explain these

Canoski et al. specify *max 100 epochs, early-stopping patience 20*. This notebook uses
**max 25 and patience 5** so a run fits one free Colab session. Optimiser, learning rates,
schedule, augmentation, loss, label smoothing, batch size and seed all follow the paper.
`results.json` records whether the run early-stopped, so you can state honestly whether the
shortened budget was ever reached.

Mixed-precision training is enabled on GPU for speed. It does not change the protocol.

## 1 · Environment check

In [ ]:
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=30).stdout.strip()
except (FileNotFoundError, subprocess.SubprocessError):
    gpu = ""
if gpu:
    print("GPU:", gpu)
else:
    print("!! NO GPU DETECTED")
    print("!! Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all")
    print("!! (it will still run on CPU, but expect hours rather than minutes)")

In [ ]:
import os, re, json, time, random, pathlib, collections, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")

## 2 · Configuration

The only cell you normally edit.

In [ ]:
CFG = dict(
    # ---- data -----------------------------------------------------------
    DATA_ROOT = "/content/grape_dataset",     # must contain train/ and test/

    # ---- partitioning ---------------------------------------------------
    # "source"   : pool everything, split by SOURCE LEAF, evaluate on originals only  <- correct
    # "naive"    : pool everything, split by IMAGE, ignore leaf identity
    # "supplied" : use the train/ and test/ folders exactly as provided
    SPLIT_MODE = "source",
    EVAL_ON_ORIGINALS_ONLY = True,   # applies to "source" only
    TRAIN_FRAC = 0.70, VAL_FRAC = 0.15, TEST_FRAC = 0.15,

    # ---- model ----------------------------------------------------------
    MODEL    = "resnet50",     # "resnet50" (matches your paper) | "efficientnet_b0" (faster)
    IMG_SIZE = 224,
    RESIZE   = 256,

    # ---- training: Canoski et al. 2026, Section 5.2 ----------------------
    BATCH_SIZE      = 64,
    HEAD_EPOCHS     = 5,      # stage 1, backbone frozen
    FINETUNE_EPOCHS = 25,     # stage 2 (paper: 100)
    PATIENCE        = 5,      # (paper: 20)
    LR_HEAD         = 1e-3,
    LR_BACKBONE     = 1e-4,
    WEIGHT_DECAY    = 1e-4,
    LR_MIN_STAGE1   = 1e-5,
    LR_MIN_STAGE2   = 1e-6,
    LABEL_SMOOTHING = 0.1,
    NUM_WORKERS     = 2,
    SEED            = 42,
    USE_AMP         = True,   # mixed precision on GPU; no effect on CPU

    # ---- calibration ----------------------------------------------------
    N_BINS  = 15,
    OUT_DIR = "/content/outputs",
)

assert CFG["SPLIT_MODE"] in ("source", "naive", "supplied"), "bad SPLIT_MODE"
assert abs(CFG["TRAIN_FRAC"] + CFG["VAL_FRAC"] + CFG["TEST_FRAC"] - 1.0) < 1e-9, "fractions must sum to 1"
os.makedirs(CFG["OUT_DIR"], exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG["SEED"])

RUN = f"{CFG['MODEL']}_{CFG['SPLIT_MODE']}"
print(f"run id: {RUN}")
print(json.dumps(CFG, indent=2))

## 3 · Load the data

Upload `grape_dataset_for_colab.zip` to the top level of your Google Drive, then run this
cell. Check `ZIP_PATH` matches the filename in Drive.

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/grape_dataset_for_colab.zip"   # <-- must match Drive

import zipfile, shutil
IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    print("not running in Colab - skipping Drive mount")

if IN_COLAB:
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(
            f"{ZIP_PATH} not found.\n"
            f"Check the file is at the TOP LEVEL of My Drive and the name matches exactly.\n"
            f"Files in My Drive: {sorted(os.listdir('/content/drive/MyDrive'))[:40]}")
    if os.path.exists(CFG["DATA_ROOT"]):
        shutil.rmtree(CFG["DATA_ROOT"])
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/")

assert os.path.isdir(CFG["DATA_ROOT"]), f"{CFG['DATA_ROOT']} missing after extraction"
print("data root:", CFG["DATA_ROOT"])
print("contains :", sorted(os.listdir(CFG["DATA_ROOT"])))

## 4 · Index the files

Two things are recovered from each filename:

* the **source leaf** — the leading UUID, shared by every copy of one physical leaf;
* whether the file is an **original photograph** or an **augmented copy** — the suffix.

Anything that does not match is counted and reported rather than silently assumed.

In [ ]:
UUID_RE = re.compile(r"^([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})___", re.I)
AUG_RE  = re.compile(r"(_flipLR|_90deg|_180deg|_270deg|_new30degFlipLR)$", re.I)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def clean_class(name):
    return name.split("___")[-1] if "___" in name else name

Rec = collections.namedtuple("Rec", "path cls leaf aug suffix partition")

records, n_no_uuid = [], 0
suffixes = collections.Counter()
root = pathlib.Path(CFG["DATA_ROOT"])

for part_dir in sorted(p for p in root.iterdir() if p.is_dir()):
    for cls_dir in sorted(p for p in part_dir.iterdir() if p.is_dir()):
        cls = clean_class(cls_dir.name)
        for f in sorted(cls_dir.iterdir()):
            if f.suffix.lower() not in IMG_EXT:
                continue
            m = UUID_RE.match(f.name)
            if m:
                leaf = m.group(1).lower()
            else:
                leaf = f"NOUUID::{f.stem}"; n_no_uuid += 1
            am = AUG_RE.search(f.stem)
            suf = am.group(1) if am else ""
            suffixes[suf] += 1
            records.append(Rec(str(f), cls, leaf, bool(am), suf, part_dir.name.lower()))

assert records, "no images found - check DATA_ROOT"
CLASSES = sorted({r.cls for r in records})
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
n_leaves = len({r.leaf for r in records})

print(f"images                 : {len(records):,}")
print(f"source leaves          : {n_leaves:,}")
print(f"original photographs   : {sum(1 for r in records if not r.aug):,}")
print(f"augmented copies       : {sum(1 for r in records if r.aug):,}")
print(f"filenames without UUID : {n_no_uuid:,}")
print(f"\nclasses ({len(CLASSES)}): {CLASSES}")
print("\naugmentation vocabulary:")
for s, n in suffixes.most_common():
    print(f"  {(s or '(original)'):<20} {n:>6,}")
print("\nper class:")
for c in CLASSES:
    img = sum(1 for r in records if r.cls == c)
    lf  = len({r.leaf for r in records if r.cls == c})
    org = sum(1 for r in records if r.cls == c and not r.aug)
    print(f"  {c:<38} {img:>6,} images  {lf:>5,} leaves  {org:>5,} originals  x{img/max(lf,1):.2f}")

if n_no_uuid:
    warnings.warn(f"{n_no_uuid} files had no UUID and were treated as their own leaf")

## 5 · Leakage audit of the supplied split

Two measurements, both about the folders **as supplied**:

1. **Leaf overlap** — of the source leaves appearing in `test/`, how many also appear in `train/`?
2. **Original-photograph leakage** — of the unmodified photographs in `test/`, how many have a
   copy of that same leaf sitting in `train/`?

The second is the sharper statement: it counts real photographs whose mirror image the model
was trained on.

In [ ]:
tr_leaves = {r.leaf for r in records if r.partition.startswith("train")}
te_leaves = {r.leaf for r in records if r.partition.startswith("test")}
test_originals = [r for r in records if r.partition.startswith("test") and not r.aug]
compromised = [r for r in test_originals if r.leaf in tr_leaves]

audit = {
    "n_images": len(records),
    "n_source_leaves": n_leaves,
    "n_originals": sum(1 for r in records if not r.aug),
    "n_augmented": sum(1 for r in records if r.aug),
    "augmentation_vocabulary": {(s or "(original)"): n for s, n in suffixes.most_common()},
    "supplied_train_leaves": len(tr_leaves),
    "supplied_test_leaves": len(te_leaves),
    "test_leaves_also_in_train": len(tr_leaves & te_leaves),
    "pct_test_leaves_also_in_train": round(100 * len(tr_leaves & te_leaves) / max(len(te_leaves), 1), 1),
    "n_test_originals": len(test_originals),
    "n_test_originals_leaf_in_train": len(compromised),
    "pct_test_originals_leaf_in_train": round(100 * len(compromised) / max(len(test_originals), 1), 1),
    "per_class_pct_test_leaves_also_in_train": {},
}

print(f"source leaves in supplied train : {len(tr_leaves):,}")
print(f"source leaves in supplied test  : {len(te_leaves):,}")
print(f"  of which also in train        : {len(tr_leaves & te_leaves):,}"
      f"  ({audit['pct_test_leaves_also_in_train']}%)")
print()
print(f"original photographs in test    : {len(test_originals):,}")
print(f"  whose leaf appears in train   : {len(compromised):,}"
      f"  ({audit['pct_test_originals_leaf_in_train']}%)")
print("\nleaf overlap by class:")
for c in CLASSES:
    t = {r.leaf for r in records if r.cls == c and r.partition.startswith("train")}
    s = {r.leaf for r in records if r.cls == c and r.partition.startswith("test")}
    pct = 100 * len(t & s) / max(len(s), 1)
    audit["per_class_pct_test_leaves_also_in_train"][c] = round(pct, 1)
    print(f"  {c:<38} {len(t & s):>5,}/{len(s):<5,} = {pct:5.1f}%")

with open(f"{CFG['OUT_DIR']}/audit.json", "w") as fh:
    json.dump(audit, fh, indent=2)
print(f"\nwritten -> {CFG['OUT_DIR']}/audit.json")

## 6 · Build the partitions

| `SPLIT_MODE` | How | What it shows |
|---|---|---|
| `source` | Split the **leaves** 70/15/15 stratified by class; evaluate on **originals only** | The honest baseline |
| `naive` | Split the **images** 70/15/15 stratified by class | What ignoring leaf identity buys you |
| `supplied` | Use `train/` and `test/` as given; carve validation out of train | What the dataset as distributed produces |

Run `source` first. The other two exist so the difference can be measured rather than asserted.

In [ ]:
set_seed(CFG["SEED"])
val_rel = CFG["VAL_FRAC"] / (CFG["VAL_FRAC"] + CFG["TEST_FRAC"])
dropped_aug = 0

if CFG["SPLIT_MODE"] == "source":
    leaf_cls = {}
    for r in records:
        leaf_cls[r.leaf] = r.cls                       # one leaf, one class
    leaves = sorted(leaf_cls)
    y = [leaf_cls[g] for g in leaves]
    g_tr, g_tmp, y_tr, y_tmp = train_test_split(
        leaves, y, train_size=CFG["TRAIN_FRAC"], stratify=y, random_state=CFG["SEED"])
    g_va, g_te, _, _ = train_test_split(
        g_tmp, y_tmp, train_size=val_rel, stratify=y_tmp, random_state=CFG["SEED"])
    g_tr, g_va, g_te = set(g_tr), set(g_va), set(g_te)

    SPLITS = {"train": [], "val": [], "test": []}
    for r in records:
        s = "train" if r.leaf in g_tr else ("val" if r.leaf in g_va else "test")
        if s != "train" and CFG["EVAL_ON_ORIGINALS_ONLY"] and r.aug:
            dropped_aug += 1        # discarded, NOT moved to train
            continue
        SPLITS[s].append(r)

elif CFG["SPLIT_MODE"] == "naive":
    y = [r.cls for r in records]
    i_tr, i_tmp, y_tr, y_tmp = train_test_split(
        range(len(records)), y, train_size=CFG["TRAIN_FRAC"], stratify=y,
        random_state=CFG["SEED"])
    i_va, i_te, _, _ = train_test_split(
        i_tmp, y_tmp, train_size=val_rel, stratify=y_tmp, random_state=CFG["SEED"])
    SPLITS = {"train": [records[i] for i in i_tr],
              "val":   [records[i] for i in i_va],
              "test":  [records[i] for i in i_te]}

else:  # supplied
    supplied_train = [r for r in records if r.partition.startswith("train")]
    SPLITS = {"test": [r for r in records if r.partition.startswith("test")]}
    y = [r.cls for r in supplied_train]
    i_tr, i_va, _, _ = train_test_split(
        range(len(supplied_train)), y,
        train_size=1 - CFG["VAL_FRAC"], stratify=y, random_state=CFG["SEED"])
    SPLITS["train"] = [supplied_train[i] for i in i_tr]
    SPLITS["val"]   = [supplied_train[i] for i in i_va]

print(f"split mode: {CFG['SPLIT_MODE']}"
      + (f"   (evaluating on originals only; {dropped_aug:,} augmented copies of "
         f"val/test leaves discarded)" if dropped_aug else ""))
print()
for s in ("train", "val", "test"):
    rs = SPLITS[s]
    org = sum(1 for r in rs if not r.aug)
    print(f"  {s:<6} {len(rs):>6,} images  {len({r.leaf for r in rs}):>5,} leaves  {org:>6,} originals")

tr_g = {r.leaf for r in SPLITS['train']}
va_g = {r.leaf for r in SPLITS['val']}
te_g = {r.leaf for r in SPLITS['test']}
print(f"\nleaf overlap  train<->val  : {len(tr_g & va_g):,}")
print(f"leaf overlap  train<->test : {len(tr_g & te_g):,}")

if CFG["SPLIT_MODE"] == "source":
    assert not (tr_g & te_g), "grouping failed: leaf in both train and test"
    assert not (tr_g & va_g), "grouping failed: leaf in both train and val"
    if CFG["EVAL_ON_ORIGINALS_ONLY"]:
        assert all(not r.aug for r in SPLITS["val"] + SPLITS["test"]), \
            "augmented image found in an evaluation partition"
        print("OK - no leaf spans two partitions; val/test contain original photographs only")
    else:
        print("OK - no leaf spans two partitions")
else:
    print("(overlap is expected and is the point of this mode)")

for s in ("train", "val", "test"):
    assert len(SPLITS[s]) > 0, f"{s} partition is empty"
    assert len({r.cls for r in SPLITS[s]}) == len(CLASSES), f"{s} is missing a class"
print("all partitions non-empty and contain every class")

## 7 · Transforms and loaders

The augmentation, resizing and normalisation below are exactly those specified in Canoski et al., Section 5.2.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(CFG["RESIZE"]),
    transforms.RandomCrop(CFG["IMG_SIZE"]),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(CFG["RESIZE"]),
    transforms.CenterCrop(CFG["IMG_SIZE"]),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class LeafDS(Dataset):
    def __init__(self, rows, tf): self.rows, self.tf = rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        with Image.open(r.path) as im:
            img = im.convert("RGB")
        return self.tf(img), CLS2IDX[r.cls]

def loader(rows, tf, shuffle):
    return DataLoader(LeafDS(rows, tf), batch_size=CFG["BATCH_SIZE"], shuffle=shuffle,
                      num_workers=CFG["NUM_WORKERS"], pin_memory=(DEVICE == "cuda"),
                      persistent_workers=CFG["NUM_WORKERS"] > 0, drop_last=False)

train_dl = loader(SPLITS["train"], train_tf, True)
val_dl   = loader(SPLITS["val"],   eval_tf,  False)
test_dl  = loader(SPLITS["test"],  eval_tf,  False)

xb, yb = next(iter(train_dl))
print(f"batch tensor {tuple(xb.shape)} dtype {xb.dtype} | labels {tuple(yb.shape)}")
print(f"value range  [{xb.min():.2f}, {xb.max():.2f}]  (normalised, so negatives are expected)")

## 8 · Model and loss

Class weights come from the **training partition only** — no information from validation or test enters the loss.

In [ ]:
def build_model(name, n_classes):
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        return m, ("fc",)
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        return m, ("classifier",)
    raise ValueError(f"unknown model: {name}")

model, HEAD = build_model(CFG["MODEL"], len(CLASSES))
model = model.to(DEVICE)

def head_params():
    return [p for n, p in model.named_parameters() if n.startswith(HEAD)]
def backbone_params():
    return [p for n, p in model.named_parameters() if not n.startswith(HEAD)]

assert head_params(), "no head parameters matched - check HEAD prefix"
assert backbone_params(), "no backbone parameters matched"

counts = collections.Counter(r.cls for r in SPLITS["train"])
n, k = len(SPLITS["train"]), len(CLASSES)
weights = torch.tensor([n / (k * counts[c]) for c in CLASSES],
                       dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=CFG["LABEL_SMOOTHING"])

print(f"{CFG['MODEL']}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, "
      f"{len(head_params())} head tensors")
print("train class counts :", {c: counts[c] for c in CLASSES})
print("class weights      :", {c: round(float(w), 3) for c, w in zip(CLASSES, weights)})

## 9 · Train

Stage 1 trains the new classifier head with the backbone frozen. Stage 2 unfreezes everything and fine-tunes with a lower learning rate on the backbone.

In [ ]:
AMP = CFG["USE_AMP"] and DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=AMP)

def run_epoch(loader_, train, optimiser=None):
    model.train(train)
    tot_loss = correct = seen = 0
    with torch.set_grad_enabled(train):
        for x, y in loader_:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", enabled=AMP):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimiser.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimiser)
                scaler.update()
            tot_loss += loss.item() * y.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            seen     += y.size(0)
    return tot_loss / seen, correct / seen

CKPT = f"{CFG['OUT_DIR']}/best_{RUN}.pt"
history, best_val, bad, stopped_early = [], float("inf"), 0, False
t0 = time.time()

# ---- stage 1 : head only ------------------------------------------------
for p in backbone_params():
    p.requires_grad_(False)
opt = torch.optim.AdamW(head_params(), lr=CFG["LR_HEAD"], weight_decay=CFG["WEIGHT_DECAY"])
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["HEAD_EPOCHS"],
                                                 eta_min=CFG["LR_MIN_STAGE1"])
print("stage 1 - backbone frozen")
for ep in range(1, CFG["HEAD_EPOCHS"] + 1):
    tl, ta = run_epoch(train_dl, True, opt)
    vl, va = run_epoch(val_dl, False)
    sch.step()
    history.append(dict(stage=1, epoch=ep, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va))
    star = ""
    if vl < best_val:
        best_val, star = vl, "  *"
        torch.save(model.state_dict(), CKPT)
    print(f"  [{ep}/{CFG['HEAD_EPOCHS']}] train {tl:.4f}/{ta:.4f}  val {vl:.4f}/{va:.4f}{star}")

# ---- stage 2 : full fine-tune ------------------------------------------
for p in model.parameters():
    p.requires_grad_(True)
opt = torch.optim.AdamW(
    [{"params": backbone_params(), "lr": CFG["LR_BACKBONE"]},
     {"params": head_params(),     "lr": CFG["LR_HEAD"]}],
    weight_decay=CFG["WEIGHT_DECAY"])
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["FINETUNE_EPOCHS"],
                                                 eta_min=CFG["LR_MIN_STAGE2"])
print("\nstage 2 - full fine-tune")
for ep in range(1, CFG["FINETUNE_EPOCHS"] + 1):
    tl, ta = run_epoch(train_dl, True, opt)
    vl, va = run_epoch(val_dl, False)
    sch.step()
    history.append(dict(stage=2, epoch=ep, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va))
    star = ""
    if vl < best_val:
        best_val, bad, star = vl, 0, "  *"
        torch.save(model.state_dict(), CKPT)
    else:
        bad += 1
    print(f"  [{ep}/{CFG['FINETUNE_EPOCHS']}] train {tl:.4f}/{ta:.4f}  val {vl:.4f}/{va:.4f}{star}")
    if bad >= CFG["PATIENCE"]:
        print(f"  early stop - no validation improvement for {CFG['PATIENCE']} epochs")
        stopped_early = True
        break

model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
mins = (time.time() - t0) / 60
print(f"\ntrained in {mins:.1f} min | best val loss {best_val:.4f} | early stop: {stopped_early}")

## 10 · Test-set performance

In [ ]:
@torch.no_grad()
def collect_logits(loader_):
    model.eval()
    L, Y = [], []
    for x, y in loader_:
        L.append(model(x.to(DEVICE)).float().cpu())
        Y.append(y)
    return torch.cat(L), torch.cat(Y)

test_logits, test_y = collect_logits(test_dl)
val_logits,  val_y  = collect_logits(val_dl)

probs   = F.softmax(test_logits, 1).numpy()
preds   = probs.argmax(1)
y       = test_y.numpy()
confs   = probs.max(1)
correct = (preds == y).astype(float)
accuracy = float(correct.mean())

print(f"TEST ACCURACY  {accuracy:.4f}   ({int(correct.sum()):,}/{len(y):,})\n")
print("per class:")
for i, c in enumerate(CLASSES):
    m = y == i
    print(f"  {c:<38} {correct[m].mean():.4f}  (n={int(m.sum()):,})")

cm = np.zeros((len(CLASSES), len(CLASSES)), int)
for t_, p_ in zip(y, preds):
    cm[t_, p_] += 1
print("\nconfusion matrix (row = true, col = predicted):")
print(f"{'':<40}" + "".join(f"{c[:11]:>13}" for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f"  {c:<38}" + "".join(f"{v:>13,}" for v in cm[i]))

## 11 · Calibration

**Expected Calibration Error** sorts predictions into confidence bins and averages
|accuracy − confidence| across bins, weighted by how many predictions fall in each. A
perfectly calibrated classifier scores 0. A classifier that says 90% and is right 70% of the
time is overconfident, and on the reliability diagram its bars sit below the diagonal.

In [ ]:
def ece_and_bins(confs_, correct_, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    ece, rows = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (confs_ > lo) & (confs_ <= hi) if i > 0 else (confs_ >= lo) & (confs_ <= hi)
        if not m.any():
            rows.append(dict(lo=lo, hi=hi, n=0, acc=np.nan, conf=np.nan))
            continue
        acc, conf = float(correct_[m].mean()), float(confs_[m].mean())
        ece += float(m.mean()) * abs(acc - conf)
        rows.append(dict(lo=lo, hi=hi, n=int(m.sum()), acc=acc, conf=conf))
    return ece, rows

ece, bins_ = ece_and_bins(confs, correct, CFG["N_BINS"])
print(f"accuracy           {accuracy:.4f}")
print(f"mean confidence    {confs.mean():.4f}")
print(f"overconfidence     {confs.mean() - accuracy:+.4f}")
print(f"ECE                {ece:.4f}\n")
print(f"{'bin':<15}{'n':>8}{'acc':>9}{'conf':>9}{'gap':>9}")
for b in bins_:
    if b["n"]:
        print(f"[{b['lo']:.2f},{b['hi']:.2f}]{b['n']:>9,}{b['acc']:>9.3f}"
              f"{b['conf']:>9.3f}{b['conf'] - b['acc']:>+9.3f}")

### Reliability diagram and confidence histogram

In [ ]:
BLUE, RED, INK, MUTED, GRID = "#2a78d6", "#e34948", "#0b0b0b", "#52514e", "#d8d7d2"
matplotlib.rcParams.update({
    "font.size": 15, "axes.labelsize": 16, "axes.titlesize": 17,
    "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 14,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": MUTED, "text.color": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "savefig.facecolor": "white",
})

def plot_calibration(confs_, correct_, title, path):
    e, rows = ece_and_bins(confs_, correct_, CFG["N_BINS"])
    centres = np.array([(r["lo"] + r["hi"]) / 2 for r in rows])
    accs    = np.array([r["acc"] for r in rows], float)
    cfs     = np.array([r["conf"] for r in rows], float)
    ok      = ~np.isnan(accs)
    width   = 0.88 / CFG["N_BINS"]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8.5), sharex=True,
                                   gridspec_kw=dict(height_ratios=[3, 1.15], hspace=0.12))
    ax1.bar(centres[ok], np.abs(cfs[ok] - accs[ok]),
            bottom=np.minimum(accs[ok], cfs[ok]), width=width,
            color=RED, alpha=0.30, edgecolor=RED, linewidth=1.2,
            label="Calibration gap", zorder=2)
    ax1.bar(centres[ok], accs[ok], width=width, color=BLUE,
            edgecolor="white", linewidth=2, label="Accuracy", zorder=3)
    ax1.plot([0, 1], [0, 1], color=MUTED, ls="--", lw=2,
             label="Perfect calibration", zorder=4)
    ax1.set(xlim=(0, 1), ylim=(0, 1.02), ylabel="Accuracy")
    ax1.set_title(title, pad=14, weight="bold")
    ax1.grid(axis="y", color=GRID, lw=1); ax1.set_axisbelow(True)
    for s in ("top", "right"): ax1.spines[s].set_visible(False)
    ax1.legend(loc="upper left", frameon=False)
    ax1.text(0.97, 0.06, f"ECE = {e:.3f}", transform=ax1.transAxes, ha="right",
             va="bottom", fontsize=20, weight="bold", color=INK)

    ax2.hist(confs_, bins=np.linspace(0, 1, CFG["N_BINS"] + 1),
             color=BLUE, edgecolor="white", linewidth=2)
    ax2.axvline(confs_.mean(), color=RED, lw=2.5, ls="--")
    ax2.text(confs_.mean() - 0.015, ax2.get_ylim()[1] * 0.86,
             f"mean {confs_.mean():.2f}", ha="right", color=RED,
             fontsize=14, weight="bold")
    ax2.set(xlabel="Confidence (max softmax probability)", ylabel="Predictions")
    ax2.grid(axis="y", color=GRID, lw=1); ax2.set_axisbelow(True)
    for s in ("top", "right"): ax2.spines[s].set_visible(False)

    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    return e

_ = plot_calibration(confs, correct,
                     f"Before calibration - {CFG['MODEL']}, {CFG['SPLIT_MODE']} split",
                     f"{CFG['OUT_DIR']}/reliability_before_{RUN}.png")

## 12 · Temperature scaling

A single scalar *T*, fitted on the **validation** set by minimising negative log-likelihood,
then applied to the test logits. The test partition is never used to fit it. Dividing the
logits by *T* cannot change which class is largest, so accuracy is unchanged by construction —
only the confidence attached to each prediction moves.

In [ ]:
log_T = torch.zeros(1, requires_grad=True)     # T = exp(log_T) keeps T positive
opt_T = torch.optim.LBFGS([log_T], lr=0.1, max_iter=100)
nll = nn.CrossEntropyLoss()

def closure():
    opt_T.zero_grad()
    loss = nll(val_logits / torch.exp(log_T), val_y)
    loss.backward()
    return loss

opt_T.step(closure)
T = float(torch.exp(log_T).detach())

probs_T   = F.softmax(test_logits / T, 1).numpy()
confs_T   = probs_T.max(1)
correct_T = (probs_T.argmax(1) == y).astype(float)
ece_T, _  = ece_and_bins(confs_T, correct_T, CFG["N_BINS"])

print(f"fitted temperature T = {T:.4f}  "
      f"({'softening overconfident' if T > 1 else 'sharpening underconfident'} probabilities)\n")
print(f"{'':<18}{'before':>10}{'after':>10}")
print(f"{'accuracy':<18}{accuracy:>10.4f}{correct_T.mean():>10.4f}")
print(f"{'mean confidence':<18}{confs.mean():>10.4f}{confs_T.mean():>10.4f}")
print(f"{'ECE':<18}{ece:>10.4f}{ece_T:>10.4f}")
assert abs(correct_T.mean() - accuracy) < 1e-9, "temperature scaling must not change accuracy"

_ = plot_calibration(confs_T, correct_T,
                     f"After temperature scaling (T = {T:.2f})",
                     f"{CFG['OUT_DIR']}/reliability_after_{RUN}.png")

## 13 · Save and download

In [ ]:
results = dict(
    run=RUN,
    config=CFG,
    classes=CLASSES,
    audit=audit,
    dropped_augmented_from_eval=int(dropped_aug),
    split_sizes={s: len(r) for s, r in SPLITS.items()},
    split_leaves={s: len({x.leaf for x in r}) for s, r in SPLITS.items()},
    split_originals={s: int(sum(1 for x in r if not x.aug)) for s, r in SPLITS.items()},
    epochs_run=len(history),
    stopped_early=bool(stopped_early),
    minutes=round(mins, 2),
    best_val_loss=float(best_val),
    test_accuracy=accuracy,
    per_class_accuracy={c: float(correct[y == i].mean()) for i, c in enumerate(CLASSES)},
    confusion_matrix=cm.tolist(),
    mean_confidence=float(confs.mean()),
    ece=float(ece),
    temperature=float(T),
    ece_after_temperature=float(ece_T),
    mean_confidence_after=float(confs_T.mean()),
    history=history,
)
res_path = f"{CFG['OUT_DIR']}/results_{RUN}.json"
with open(res_path, "w") as fh:
    json.dump(results, fh, indent=2)
np.savez(f"{CFG['OUT_DIR']}/test_outputs_{RUN}.npz",
         logits=test_logits.numpy(), labels=y, temperature=np.array([T]))

bar = "=" * 62
print(bar); print(" HEADLINE NUMBERS"); print(bar)
print(f" split mode            {CFG['SPLIT_MODE']}")
print(f" model                 {CFG['MODEL']}")
print(f" test images           {len(y):,}")
print(f" test accuracy         {accuracy:.4f}")
print(f" mean confidence       {confs.mean():.4f}")
print(f" ECE                   {ece:.4f}")
print(f" temperature           {T:.3f}")
print(f" ECE after scaling     {ece_T:.4f}")
print(bar)
print(f"\nwritten to {CFG['OUT_DIR']}")

if IN_COLAB:
    from google.colab import files
    for f in (f"reliability_before_{RUN}.png", f"reliability_after_{RUN}.png",
              f"results_{RUN}.json", "audit.json"):
        p = f"{CFG['OUT_DIR']}/{f}"
        if os.path.exists(p):
            try:
                files.download(p)
            except Exception as e:
                print(f"could not auto-download {f} ({e}) - use the file browser on the left")

---

## What to do with the output

**Run 1 — `SPLIT_MODE = "source"`.** Your honest baseline. This is the number for your slide.

**Run 2 — `SPLIT_MODE = "supplied"`.** Change it in Section 2, then **Runtime → Restart
session and run all**. This reproduces what the dataset produces when used as distributed.

The difference between those two accuracies is the cost of the leakage, measured rather than
asserted. `"naive"` is a third point if you want it.

### Reading the result honestly

- A high accuracy on `supplied` and a lower one on `source` means **the supplied split is
  optimistic**. That is a statement about this dataset.
- It is **not**, on its own, a statement about anyone's published accuracy. Keep that
  distinction until Shu confirms where the dataset came from.
- If ECE is very small on `source`, that is not a null result — it is the expected consequence
  of a model with almost no errors to be miscalibrated about, and it is the reason the project
  moves on to degraded images, where errors actually occur.